In [0]:
# 1 VM - 8GB RAM and 512 GB ROM
# cluster - combination of multiple Nodes(VM) interconnected with each other
# Node = VM in spark
# cluster - 8 nodes
# 1 node - 4 core , 8GBRAM , 512 GB
# pandas -  always run in single Node
# spark will execute your code in 8 nodes( parallel processing)
# driver
# DAG scheduler(DIrected acyclic graph)
# Cluster manager
# Worker Nodes(Executors)

In [0]:
import pyspark
from pyspark.sql.functions import *

In [0]:
import pandas as pd
df = pd.read_csv('/Volumes/workspace/default/datasets/Flight_data.csv')
# df.display() 
# spark submit

In [0]:
type(df)

In [0]:
df = spark.read.csv('/Volumes/workspace/default/datasets/Flight_data.csv', header=True)
# df.display() 
# spark submit

In [0]:
type(df)

In [0]:
%sql
select * from workspace.default.student
--spark core API

In [0]:
df_emp = spark.sql('select * from workspace.default.student')
df_emp.display()
# spark sql

In [0]:
from pyspark import SparkContext
sc = SparkContext("local", "My First RDD")
# sc.show()

In [0]:
# spark 1.0 - spark contxt
# spark 2.0 spark session
# spark session = spark context + sql context

In [0]:
sc.parallelize([(1,2,3)])

In [0]:
# 1 cluster --8 nodes
# 1 node - 8GB 512 GB 
# each node we got only 1GB data
# rest 7 GB

# spark underperforms for low volume data

## Spark session

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession

In [0]:
Spark = SparkSession.builder.appName("SparkByExamples.com").getOrCreate() 


In [0]:
df = spark.createDataFrame([('A',10,23),('B',11,28),('C',12,31)],schema = ['Name','ID','Age'])
df_new = df.groupby('Name').agg(sum(col('age')))
#df_new.display()

In [0]:
df_new.display()

## Spark dataframe

In [0]:
df_name = spark.createDataFrame([('Jiten',32,'M'),('Sarojini',25,'F'),('Lipsa',24,'F')],schema = ['Name','age','gender'])
df_name.display()

In [0]:
df_name = spark.createDataFrame([{'Name':'Jiten','age':32,'gender':'M'}])
df_name.display()

In [0]:
df_name.dtypes

In [0]:
df_name.schema

In [0]:
df_n = df_name.select('Name',col("age").cast(StringType()))
#df_n.display()

In [0]:
schema = StructType([
    StructField('Org_name', StringType(),nullable=True),
    StructField('Emp_id', StringType(),nullable=True),
    StructField('DOJ', DateType(),nullable=True)
]
)

In [0]:
from datetime import datetime,timedelta
datetime.now().date()

In [0]:
datetime.now()-timedelta(1)
datetime(2025, 4, 11, 2, 45).date()

In [0]:
spark.createDataFrame([('LTI',None,datetime(2024, 7, 15).date())],schema=schema).display()

In [0]:
spark.createDataFrame([{'name':['sital','ravi'],'age':[12,34]}]).select(explode(col('name')).alias('name'),explode(col('age')).alias('age')).distinct().display()

In [0]:
df_flight = spark.read.csv('/Volumes/workspace/default/datasets/Flight_data.csv')
# df_flight.display()

In [0]:
df_flight = spark.read.format('csv').option('header', True).load('/Volumes/workspace/default/datasets/Flight_data.csv')
# df_flight.display() # inferschema

In [0]:
df_flight.schema

In [0]:
schema = StructType([StructField('_c0', StringType(), True), StructField('Airline', StringType(), True), StructField('Date_of_Journey', StringType(), True), StructField('Source', StringType(), True), StructField('Destination', StringType(), True), StructField('Route', StringType(), True), StructField('Dep_Time', StringType(), True), StructField('Arrival_Time', StringType(), True), StructField('Duration', StringType(), True), StructField('Total_Stops', StringType(), True), StructField('Additional_Info', StringType(), True), StructField('Price', IntegerType(), True)])

In [0]:
df_flight.select('Airline','Source','Destination').display()

In [0]:
df_flight = spark.read.format('csv').option('header', True).schema(schema).load('/Volumes/workspace/default/datasets/Flight_data.csv')
# df_flight.display()

In [0]:
emp_schema = StructType([StructField('EMPNO', StringType(), True), StructField('ENAME', StringType(), True), StructField('JOB', StringType(), True), StructField('MGR', StringType(), True), StructField('HIREDATE', StringType(), True), StructField('SAL', StringType(), True), StructField('COMM', IntegerType(), True), StructField('DEPTNO', StringType(), True),StructField('_corrupt_record', StringType(), True)])

In [0]:
df_emp = spark.read.format("csv").option("header", True).option("mode", "PERMISSIVE").schema(
    emp_schema
).option("columnNameOfCorruptRecord", "_corrupt_record").load(
    "/Volumes/workspace/default/test_practice/emp_new.csv"
)
df_emp.filter(col('_corrupt_record').isNotNull()).display()

In [0]:
spark.read.format("csv").option("header", True).option("mode", "DROPMALFORMED").schema(
    emp_schema
).load(
    "/Volumes/workspace/default/test_practice/emp_new.csv"
).display()

In [0]:
spark.read.format("csv").option("header", True).option("mode", "FAILFAST").schema(
    emp_schema
).load(
    "/Volumes/workspace/default/test_practice/emp_new.csv"
).display()

In [0]:
df_emp.select([col for col in df_emp.columns if col!='_corrupt_record']).display()

In [0]:
df_emp.select([count(when(col(c).isNull(),c)).alias(c) for c in df_emp.columns]).display()


In [0]:
df_emp.withColumn('COMM', when(col('COMM').isNull() , lit(0)).otherwise(col('COMM'))).display()

In [0]:
df_emp.filter(col('JOB')=='MANAGER').display()

In [0]:
df_emp.filter((col('JOB')!='MANAGER') & (col('SAL')>1500) ).display()

In [0]:
df_emp.filter((col('JOB').isin('MANAGER','ANALYST')) & (col('SAL')>1500) ).display()

In [0]:
df_emp.withColumn('JOB', lower(col('JOB'))).select('EMPNO','ENAME','JOB','HIREDATE','SAL').display()

In [0]:
df_emp.withColumn('SAL', col('SAL').cast(IntegerType())).display()